# Structure-aware context residual: Phase 1 feasibility audit

This CPU-only notebook checks exact AlphaFold DB coverage and candidate-site pLDDT for the context-valid released training cohort. It fits no model and never accesses the released independent or external test sets. A structural-residual architecture is designed only if the frozen coverage gates pass.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

GITHUB_TOKEN = secret('GITHUB_TOKEN')
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
askpass = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) echo "$GITHUB_TOKEN";; esac\n')
    askpass.chmod(0o700)
    git_env.update({'GIT_ASKPASS': str(askpass), 'GITHUB_TOKEN': GITHUB_TOKEN})

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--recurse-submodules', 'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git', str(REPO_ROOT)], check=True, env=git_env)
    else:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True, env=git_env)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'], check=True, env=git_env)
finally:
    askpass.unlink(missing_ok=True)
os.chdir(REPO_ROOT)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
import json, requests, numpy, torch
print('CPU-only audit; a GPU is not required.')
print('requests:', requests.__version__, 'NumPy:', numpy.__version__, 'PyTorch:', torch.__version__)

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

In [ ]:
CONFIG = Path('experiment/configs/structure_context_feasibility_v1.json')
SEQUENCE_CACHE = Path('/content/drive/MyDrive/MMUbiPred-research/context/uniprot_training_sequences.json')
OUTPUT_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/structure/structure-context-feasibility-v1')
METADATA_CACHE = OUTPUT_DIR / 'alphafold_training_site_metadata.json'
REPORT_PATH = OUTPUT_DIR / 'structure_feasibility.json'
FEATURES_PATH = OUTPUT_DIR / 'site_structure_confidence.npz'

if not SEQUENCE_CACHE.exists():
    raise FileNotFoundError('Copy uniprot_training_sequences.json into /content/drive/MyDrive/MMUbiPred-research/context/')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Persistent retrieval cache:', METADATA_CACHE)
print('Frozen feasibility report:', REPORT_PATH)

## Retrieve or resume AlphaFold metadata and site confidence

The cache is written atomically after every 25 proteins. A disconnection loses at most the currently active requests. Re-execution continues retrieval until `retrieval_complete` is reported; the persistent cache is retained between sessions.

In [ ]:
run_live([
    sys.executable, '-u',
    'experiment/scripts/fetch_alphafold_site_confidence.py',
    '--data-dir', 'replication/MMUbiPred',
    '--sequence-cache', SEQUENCE_CACHE,
    '--output', METADATA_CACHE,
    '--max-workers', '6',
    '--checkpoint-every', '25',
])

## Freeze the training-only feasibility audit

This step runs once after retrieval completes. It does not download coordinates or train the proposed graph residual.

In [ ]:
if not REPORT_PATH.exists():
    run_live([
        sys.executable, '-u',
        'experiment/scripts/audit_structure_feasibility.py',
        '--config', CONFIG,
        '--metadata-cache', METADATA_CACHE,
        '--output-dir', OUTPUT_DIR,
    ])
else:
    print('Existing frozen feasibility report found; it was not overwritten.')

report = json.loads(REPORT_PATH.read_text())
display({
    'status': report['status'],
    'training_authorized': report['training_authorized'],
    'coverage_checks': report['coverage_checks'],
    'coverage': report['coverage'],
    'site_plddt': report['site_plddt'],
    'reason_counts': report['reason_counts'],
    'released_independent_or_external_test_accessed': report['released_independent_or_external_test_accessed'],
})

In [ ]:
display({
    'reason_counts_by_label': report['reason_counts_by_label'],
    'low_confidence_policy': report['low_confidence_policy'],
    'site_feature_artifact': report['site_feature_artifact'],
    'labels_used_for_model_fitting_or_selection': report['labels_used_for_model_fitting_or_selection'],
})

## Feasibility decision record

The two displayed dictionaries form the feasibility decision record. Coordinate retrieval and structural-residual training remain unauthorized until the coverage and class-balance gates pass review.